# 01 - Schema Understanding

**Task 1, Step 1-2:** Load the unified dataset and understand its structure before making any changes.

Covers:
- Loading all three raw files
- The 4 `record_type` values and what each means
- Valid categorical values (from `reference_codes.csv`)
- How `impact_link` rows connect to `event` rows via `parent_id`
- Why `event` rows have `category` but no `pillar` (the "don't pre-interpret" design principle)


In [16]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

RAW_DIR = "../data/raw"


## 1. Load all three datasets

In [17]:
import sys
sys.path.insert(0, '..')
from src.data_loader import load_dataset, validate_schema

# Task 1 works with the raw starter files (enrichment happens later in this notebook's flow)
df_data, df_impact, df_ref = load_dataset("../data", use_enriched=False)

print(f"ethiopia_fi_unified_data: {df_data.shape}")
print(f"impact_links:             {df_impact.shape}")
print(f"reference_codes:          {df_ref.shape}")

ethiopia_fi_unified_data: (43, 34)
impact_links:             (14, 35)
reference_codes:          (71, 4)


## 2. The four `record_type` values

| record_type | category column | pillar column |
|---|---|---|
| observation | empty | filled — dimension measured |
| target | empty | filled — dimension of the goal |
| event | filled — event type | empty (never pre-assigned) |
| impact_link | empty | filled — pillar of the *affected* indicator |


In [18]:
print("record_type counts (main sheet):")
print(df_data['record_type'].value_counts())
print()
print("impact_links sheet record_type check (should be all impact_link):")
print(df_impact['record_type'].value_counts())


record_type counts (main sheet):
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

impact_links sheet record_type check (should be all impact_link):
record_type
impact_link    14
Name: count, dtype: int64


In [19]:
events = df_data[df_data['record_type'] == 'event']
obs_and_targets = df_data[df_data['record_type'].isin(['observation', 'target'])]

# Use the shared validator from src/ instead of duplicating checks inline
result = validate_schema(df_data, df_ref)
if result['passed']:
    print("Schema rule check passed: events have no pillar, observations/targets have no category, all codes valid.")
else:
    print("Schema issues found:")
    for issue in result['issues']:
        print(" -", issue)

Schema rule check passed: events have no pillar, observations/targets have no category, all codes valid.


## 3. Valid categorical values (`reference_codes.csv`)

In [20]:
for field, group in df_ref.groupby('field'):
    codes = group['code'].tolist()
    print(f"{field} ({len(codes)}): {codes}")


category (10): ['product_launch', 'market_entry', 'market_exit', 'policy', 'regulation', 'infrastructure', 'partnership', 'milestone', 'economic', 'pricing']
confidence (4): ['high', 'medium', 'low', 'estimated']
evidence_basis (4): ['empirical', 'literature', 'theoretical', 'expert']
gender (3): ['all', 'male', 'female']
impact_direction (4): ['increase', 'decrease', 'stabilize', 'mixed']
impact_magnitude (4): ['high', 'medium', 'low', 'negligible']
indicator_direction (3): ['higher_better', 'lower_better', 'neutral']
location (3): ['national', 'urban', 'rural']
pillar (7): ['ACCESS', 'USAGE', 'QUALITY', 'AFFORDABILITY', 'TRUST', 'DEPTH', 'GENDER']
record_type (6): ['observation', 'event', 'impact_link', 'target', 'baseline', 'forecast']
relationship_type (4): ['direct', 'indirect', 'enabling', 'constraining']
source_type (8): ['survey', 'operator', 'regulator', 'policy', 'news', 'research', 'calculated', 'field']
value_type (11): ['percentage', 'count', 'currency_etb', 'currency_usd'

In [21]:
def validate_codes(df, column, field_name, ref=df_ref):
    """Flag any values in `column` that aren't in reference_codes for `field_name`."""
    valid = set(ref.loc[ref['field'] == field_name, 'code'])
    seen = set(df[column].dropna().unique())
    invalid = seen - valid
    if invalid:
        print(f"[{column}] INVALID values found: {invalid}")
    else:
        print(f"[{column}] all values valid ({len(seen)} distinct)")

validate_codes(df_data, 'record_type', 'record_type')
validate_codes(events, 'category', 'category')
validate_codes(df_data, 'pillar', 'pillar')
validate_codes(df_data, 'confidence', 'confidence')
validate_codes(df_data, 'source_type', 'source_type')
validate_codes(df_impact, 'relationship_type', 'relationship_type')
validate_codes(df_impact, 'impact_direction', 'impact_direction')
validate_codes(df_impact, 'impact_magnitude', 'impact_magnitude')
validate_codes(df_impact, 'evidence_basis', 'evidence_basis')


[record_type] all values valid (3 distinct)
[category] all values valid (7 distinct)
[pillar] all values valid (4 distinct)
[confidence] all values valid (2 distinct)
[source_type] all values valid (7 distinct)
[relationship_type] all values valid (3 distinct)
[impact_direction] all values valid (2 distinct)
[impact_magnitude] all values valid (3 distinct)
[evidence_basis] all values valid (3 distinct)


## 4. Events: `category` filled, `pillar` empty — by design

Events are deliberately **not** pre-assigned to a pillar, because a single event
(e.g. Telebirr) can affect multiple pillars at once. Interpretation happens only
through `impact_link` records.


In [22]:
events[['record_id', 'category', 'indicator', 'observation_date', 'source_name']]\
    .sort_values('observation_date')

,record_id,category,indicator,observation_date,source_name
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17,Ethio Telecom
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01,NBE
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01,News
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01,Safaricom
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01,NIDP
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29,NBE
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01,EthSwitch
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27,EthSwitch
42,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15,News
40,EVT_0008,infrastructure,EthioPay Instant Payment System Launch,2025-12-18,NBE/EthSwitch


## 5. How `impact_link` connects to `event` via `parent_id`

Each `impact_link` row's `parent_id` points to an `event`'s `record_id`.
One event can fan out into multiple impact_links (different pillars/indicators affected).


In [23]:
events_slim = events[['record_id', 'category', 'indicator']].rename(
    columns={'record_id': 'event_id', 'category': 'event_category', 'indicator': 'event_name'}
)

linked = df_impact.merge(events_slim, left_on='parent_id', right_on='event_id')

linked_view = linked[[
    'parent_id', 'event_name', 'event_category',
    'pillar', 'related_indicator', 'impact_direction',
    'impact_magnitude', 'lag_months', 'evidence_basis'
]]

linked_view.sort_values('parent_id')

,parent_id,event_name,event_category,pillar,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis
0,EVT_0001,Telebirr Launch,product_launch,ACCESS,ACC_OWNERSHIP,increase,high,12,literature
1,EVT_0001,Telebirr Launch,product_launch,USAGE,USG_TELEBIRR_USERS,increase,high,3,empirical
2,EVT_0001,Telebirr Launch,product_launch,USAGE,USG_P2P_COUNT,increase,high,6,empirical
3,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,ACCESS,ACC_4G_COV,increase,medium,12,empirical
4,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,12,literature
5,EVT_0003,M-Pesa Ethiopia Launch,product_launch,USAGE,USG_MPESA_USERS,increase,high,3,empirical
6,EVT_0003,M-Pesa Ethiopia Launch,product_launch,ACCESS,ACC_MM_ACCOUNT,increase,medium,6,theoretical
7,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,ACCESS,ACC_OWNERSHIP,increase,medium,24,literature
8,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,GENDER,GEN_GAP_ACC,decrease,medium,24,literature
9,EVT_0005,Foreign Exchange Liberalization,policy,AFFORDABILITY,AFF_DATA_INCOME,increase,high,3,empirical


In [24]:
from src.data_loader import event_impact_coverage

# Reusable function from src/ instead of inline groupby/merge logic
event_coverage = event_impact_coverage(df_data, df_impact)

print("Events with ZERO impact_links (candidates for enrichment):")
print(event_coverage[event_coverage['n_impact_links'] == 0])
print()
event_coverage

Events with ZERO impact_links (candidates for enrichment):
   record_id                            indicator  n_impact_links
38  EVT_0006  P2P Transaction Count Surpasses ATM               0
41  EVT_0009              NFIS-II Strategy Launch               0



,record_id,indicator,n_impact_links
38,EVT_0006,P2P Transaction Count Surpasses ATM,0
41,EVT_0009,NFIS-II Strategy Launch,0
37,EVT_0005,Foreign Exchange Liberalization,1
40,EVT_0008,EthioPay Instant Payment System Launch,1
42,EVT_0010,Safaricom Ethiopia Price Increase,1
34,EVT_0002,Safaricom Ethiopia Commercial Launch,2
35,EVT_0003,M-Pesa Ethiopia Launch,2
36,EVT_0004,Fayda Digital ID Program Rollout,2
39,EVT_0007,M-Pesa EthSwitch Integration,2
33,EVT_0001,Telebirr Launch,3


## 6. Quick indicator + pillar inventory (sets up Step 4 exploration)

In [25]:
obs = df_data[df_data['record_type'] == 'observation']

print("Observation count by pillar:")
print(obs['pillar'].value_counts())
print()
print("Observation count by indicator_code:")
print(obs['indicator_code'].value_counts())
print()
print("Date range of observations:", obs['observation_date'].min(), "to", obs['observation_date'].max())


Observation count by pillar:
pillar
ACCESS           14
USAGE            11
GENDER            4
AFFORDABILITY     1
Name: count, dtype: int64

Observation count by indicator_code:
indicator_code
ACC_OWNERSHIP         6
ACC_FAYDA             3
ACC_4G_COV            2
USG_P2P_COUNT         2
GEN_GAP_ACC           2
ACC_MM_ACCOUNT        2
USG_MPESA_USERS       1
GEN_MM_SHARE          1
AFF_DATA_INCOME       1
USG_ACTIVE_RATE       1
USG_MPESA_ACTIVE      1
USG_CROSSOVER         1
USG_TELEBIRR_VALUE    1
USG_TELEBIRR_USERS    1
USG_ATM_VALUE         1
USG_ATM_COUNT         1
USG_P2P_VALUE         1
ACC_MOBILE_PEN        1
GEN_GAP_MOBILE        1
Name: count, dtype: int64

Date range of observations: 2014-12-31 00:00:00 to 2025-12-31 00:00:00
